📅 **论文年份 (Year):2020 年**  
*Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks — Lewis et al.*

# Paper 29: Retrieval-Augmented Generation for Knowledge-Intensive Tasks(面向知识密集型任务的检索增强生成)
## Patrick Lewis, Ethan Perez, Aleksandra Piktus, et al., Meta AI (2020)(Patrick Lewis、Ethan Perez、Aleksandra Piktus 等,Meta AI,2020)

### RAG: Retrieval-Augmented Generation(RAG:检索增强生成)

Combine dense retrieval (DPR) with seq2seq generation (BART). Best of both worlds: external knowledge + powerful generation!

将稠密检索(dense retrieval,DPR)与序列到序列生成(seq2seq,BART)相结合。两全其美:外部知识 + 强大的生成能力!

## 📖 论文导读

**🎯 这篇文章想解决什么问题(目的):** 大语言模型就像一个"闭卷考试"的学生:知识全靠训练时死记硬背在参数里,遇到没背过或过时的知识就容易"一本正经地胡说八道",而且很难说清答案的出处。这篇论文想解决的正是这个问题:如何让模型在回答知识密集型问题(如开放域问答)时,能随时查阅外部资料,而不是只靠记忆?

**💡 主要贡献:** 论文提出了 RAG(检索增强生成)框架,把"查资料"和"写答案"合成一个整体:先用检索器从海量文档(如维基百科)中找到相关段落,再让生成模型参考这些段落来作答,相当于把闭卷考试变成了开卷考试。更关键的是,检索和生成两部分可以端到端一起训练,让模型学会"该查什么"和"怎么用查到的内容"。

**🔧 方法:** 具体做法是用 DPR(稠密检索器)把问题和文档都变成向量,通过向量相似度快速找出最相关的 top-k 篇文档;再用 BART(序列到序列生成模型)结合原问题和检索到的文档生成答案。论文给出两种变体:RAG-Sequence 整个回答只参考同一批文档,RAG-Token 则允许每个词参考不同的文档,并对多篇文档的结果做加权综合(边缘化)。

**🌟 意义:** 想更新知识,只需换掉资料库,无需重新训练模型——这让答案更准、更新、更可追溯来源。今天几乎所有"能查资料的 AI 助手"、企业知识库问答、客服机器人背后都是 RAG 思想,它已成为大模型应用中最普及的架构之一,这篇论文正是这一切的起点。

## 🎯 核心结论 (Key Takeaways)

- **检索让"闭卷"变"开卷",效果提升巨大**:论文在开放域问答上证明,给 BART 加上检索后,Natural Questions 的精确匹配率从 27.0% 跃升到 44.5%(RAG-Sequence),WebQuestions 从 27.6% 升到 45.2%——同样的生成模型,能查资料就几乎翻倍。
- **RAG = 检索器 (DPR) + 生成器 (BART),端到端联合训练**:先用向量相似度从文档库找出 top-k 篇相关文档,再让生成器参考它们作答;训练时梯度同时更新生成器和查询编码器,模型自己学会"该查什么、怎么用"。
- **两种变体差别在"边缘化"粒度,实测效果接近**:RAG-Sequence 整句答案共用同一批文档权重(NQ 上 44.5%),RAG-Token 每个词单独对文档加权(44.1%)。本 notebook 的热力图直观展示了这一区别——前者每列权重完全相同,后者逐列变化、可混合多篇文档的信息。
- **本 notebook 用纯 NumPy 拼出完整流程**:SimpleRetriever 用向量点积 + softmax 得到文档概率 P(z|x),SimpleGenerator 计算 P(y|x,z),再按公式 P(y|x) = Σ P(z|x)·P(y|x,z) 组合;并在一个 5 条知识、3 组问答的迷你知识库上演示了"问埃菲尔铁塔哪年建成→检索到对应文档→生成 1889"的完整链路。
- **RAG 的真正价值是知识可换、可溯源**:想更新知识只需替换文档索引,无需重新训练模型,还能查看引用的文档来核对答案;代价是检索出错就会答错,且检索带来额外延迟。
- **一句话带走**:与其让模型死记硬背,不如教它查资料——"检索概率加权生成概率"这个简单公式,就是今天所有 AI 知识库问答、企业客服机器人的源头。


## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为:想让模型懂更多知识,直觉就是把参数做大、把知识"背"进模型里。** 但这篇论文发现,给模型"外挂一个图书馆"现查现答效果好得多:同样的 BART 生成器,加上检索后 Natural Questions 精确匹配率从 27.0% 跃升到 44.5%,WebQuestions 从 27.6% 升到 45.2%——而且换知识只需替换文档索引,不用重新训练模型。
- **常识认为:"检索"是查找 + 排序,是个不可导的离散操作,没法和神经网络一起端到端训练。** 但 RAG 的妙招是把检索到的文档当作**隐变量**做边际化:P(y|x) = Σ_z P(z|x)·P(y|x,z),梯度可以顺着文档概率 P(z|x) 一路传回查询编码器。本笔记本 `RAGSequence` / `RAGToken` 的 forward 里那步"按 P(z|x) 加权求和",就是这个让不可导变可导的关键。
- **常识认为:一个问题查一次资料,整个答案用同一批文档就够了。** 但 RAG-Token 发现,生成**每一个词**时都重新给文档加权,能让不同的词分别"参考"不同的文档,把多篇资料的线索拼进同一句答案。笔记本最后的热力图正是这个对比:RAG-Sequence 每个 token 的文档权重完全相同(一列列复制),RAG-Token 则每个 token 都有自己的一套权重。
- **常识认为:答案不在检索到的文档里,模型就只能答错——抽取式模型确实如此,只能得 0 分。** 但论文发现 RAG 的生成器即使在正确答案没有出现在任何检索文档中时,仍能答对约 11.8% 的问题:它可以综合文档里的线索,再结合自身参数里的知识把答案"推"出来,而不是只会照抄原文。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库,并固定随机数种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(数值计算库,负责矩阵和向量运算)和 `matplotlib.pyplot`(画图库);
- 调用 `np.random.seed(42)` 固定随机种子——就像掷骰子前先"作弊"设定好点数顺序,这样每次重跑笔记本,所有随机生成的向量都一模一样,结果可复现。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## RAG Architecture(RAG 架构)

```
Input query (x)
    ↓
Retriever (DPR) → Top-k documents (z)
    ↓
Generator (BART) → P(y | x, z)
    ↓
Output (y)
```

**Two variants:**
- **RAG-Sequence**: Marginalize over documents for entire sequence
- **RAG-Token**: Marginalize over documents per token

**两种变体:**
- **RAG-Sequence**:对整个序列在文档上做边缘化(marginalize)
- **RAG-Token**:对每个 token 在文档上做边缘化

#### 💻 代码解读

**做什么:** 实现 RAG 的第一个核心部件——检索器(Retriever),模仿 DPR(稠密段落检索),给定一个问题,从文档库中找出最相关的前 k 篇文档,并给每篇文档一个"相关概率"。

**怎么做:**
- 定义 `softmax` 函数:把一组打分变成加起来等于 1 的概率(先减最大值防止数值溢出);
- 定义 `SimpleRetriever` 类:`encode_query` 把问题的多个词向量取平均,再经过一个矩阵 `query_encoder_W` 投影并做 L2 归一化,得到一个"问题向量"——相当于把整个问题压缩成一个坐标点;
- `retrieve` 方法:用点积(内积)计算问题向量和所有文档向量的相似度,好比比较两个箭头指向是否一致,再用 `np.argsort` 挑出相似度最高的前 k 篇,并用 softmax 把分数变成检索概率 P(z|x);
- 最后用随机生成的 10 个词的问题和 20 篇文档做测试,打印检索到的文档编号、各自概率,并验证概率之和约等于 1。

In [ ]:
def softmax(x):
    # 数值稳定技巧:先减去最大值再取指数,防止exp溢出(结果不变,因为分子分母同乘常数)
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

class SimpleRetriever:
    """Simplified dense retriever (like DPR)"""
    def __init__(self, embedding_dim):
        self.embedding_dim = embedding_dim
        self.query_encoder_W = np.random.randn(embedding_dim, embedding_dim) * 0.01
    
    def encode_query(self, query_tokens):
        """Encode query to dense vector"""
        # Simplified: just use random projection
        # 沿axis=0对所有token取平均,把变长序列压成一个定长向量:(n_tokens, dim) -> (dim,)
        query_vec = np.mean(query_tokens, axis=0)
        encoded = np.dot(self.query_encoder_W, query_vec)
        # L2 normalize
        # 归一化后点积等价于余弦相似度;+1e-8防止除以零
        return encoded / (np.linalg.norm(encoded) + 1e-8)
    
    def retrieve(self, query_embedding, document_embeddings, k=5):
        """
        Retrieve top-k documents
        Returns: indices and probabilities
        """
        # Compute similarities
        # 矩阵乘向量,一次算出查询与所有文档的相似度:(n_docs, dim) @ (dim,) -> (n_docs,)
        similarities = np.dot(document_embeddings, query_embedding)

        # Get top-k
        # argsort默认升序,[::-1]反转成降序,再切片取前k个索引
        top_k_indices = np.argsort(similarities)[::-1][:k]
        # 花式索引:用索引数组一次取出对应的k个分数
        top_k_scores = similarities[top_k_indices]

        # Convert to probabilities
        # 对应论文中的检索分布P(z|x):相似度经softmax变成文档的概率权重
        probs = softmax(top_k_scores)
        
        return top_k_indices, probs

# Test retriever
embedding_dim = 64
retriever = SimpleRetriever(embedding_dim)

# Dummy data
query_tokens = np.random.randn(10, embedding_dim)
document_embeddings = np.random.randn(20, embedding_dim)
# Normalize documents
# keepdims=True保持形状为(20, 1),才能按广播规则逐行除以各自的范数
document_embeddings = document_embeddings / (np.linalg.norm(document_embeddings, axis=1, keepdims=True) + 1e-8)

query_emb = retriever.encode_query(query_tokens)
top_indices, top_probs = retriever.retrieve(query_emb, document_embeddings, k=5)

print(f"Retrieved documents: {top_indices}")
print(f"Retrieval probabilities: {top_probs}")
print(f"Sum of probs: {np.sum(top_probs):.4f}")

## Generator (Seq2Seq)(生成器(Seq2Seq))

#### 💻 代码解读

**做什么:** 实现 RAG 的第二个核心部件——生成器(Generator),模仿 BART 这类 seq2seq 模型,计算"在给定问题 x 和检索到的文档 z 的条件下,生成目标答案 y 的概率" P(y|x,z)。

**怎么做:**
- 定义 `SimpleGenerator` 类,内部有三个随机初始化的权重矩阵:编码器 `encoder_W`、解码器 `decoder_W` 和输出层 `output_W`;
- `generate_prob` 方法先把问题和文档的词向量拼接(`np.concatenate`)后取平均,经过 tanh 得到编码器隐状态——相当于把"问题+参考资料"读一遍浓缩成一个记忆;
- 然后逐个遍历目标答案的每个词:计算解码器隐状态,与编码器隐状态相加,经过输出层和 softmax 得到词表上的概率分布,取出目标词对应的概率并累加其对数(log);
- 最后用随机数据测试,打印出整句答案的对数概率 log P(y|x,z)。数值是负的很正常——概率小于 1,取对数就是负数。

In [ ]:
class SimpleGenerator:
    """Simplified seq2seq generator (like BART)"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        
        # Encoder
        self.encoder_W = np.random.randn(hidden_dim, embedding_dim) * 0.01
        
        # Decoder
        self.decoder_W = np.random.randn(hidden_dim, embedding_dim) * 0.01
        self.output_W = np.random.randn(vocab_size, hidden_dim) * 0.01
    
    def generate_prob(self, query_tokens, doc_tokens, target_tokens):
        """
        Compute P(y | x, z) where:
        - x: query
        - z: document
        - y: target output
        """
        # Encode query + document
        # RAG的关键输入形式:把检索到的文档直接拼在查询后面一起编码
        # 形状:(n_query, dim)+(n_doc, dim) -> (n_query+n_doc, dim)
        combined = np.concatenate([query_tokens, doc_tokens], axis=0)
        # 平均池化后线性变换+tanh,得到编码器隐状态:(dim,) -> (hidden_dim,)
        encoder_hidden = np.tanh(np.dot(self.encoder_W, np.mean(combined, axis=0)))

        # Decode target
        # 用对数概率累加代替概率连乘,避免小数连乘下溢为0
        log_prob = 0
        for target_token in target_tokens:
            decoder_hidden = np.tanh(np.dot(self.decoder_W, target_token))
            
            # Combine encoder and decoder
            combined_hidden = encoder_hidden + decoder_hidden
            
            # Output distribution
            # 投影到词表空间得到logits:(vocab_size, hidden_dim) @ (hidden_dim,) -> (vocab_size,)
            logits = np.dot(self.output_W, combined_hidden)
            probs = softmax(logits)
            
            # Assume we know the target token index (simplified)
            # In reality, we'd compute cross-entropy
            target_idx = np.argmax(target_token)  # One-hot
            # 累加log P(y_t|...);+1e-8防止对0取log得到-inf
            log_prob += np.log(probs[target_idx] + 1e-8)
        
        return log_prob

# Test generator
vocab_size = 1000
generator = SimpleGenerator(vocab_size, embedding_dim, hidden_dim=128)

# Dummy tokens (embeddings)
query = np.random.randn(5, embedding_dim)
doc = np.random.randn(20, embedding_dim)
target = np.random.randn(8, embedding_dim)

log_prob = generator.generate_prob(query, doc, target)
print(f"\nLog P(y | x, z): {log_prob:.4f}")

## RAG-Sequence: Marginalize Over Documents(RAG-Sequence:在文档上边缘化)

$$
P_{RAG-Seq}(y | x) = \sum_{z \in \text{top-k}} P(z | x) \cdot P(y | x, z)
$$

Generate entire sequence with each document, then combine.

用每篇文档分别生成完整序列,然后再进行组合。

#### 💻 代码解读

**做什么:** 把检索器和生成器组装成 RAG-Sequence 模型:整句答案共用同一批检索文档,按公式 P(y|x) = Σ_z P(z|x)·P(y|x,z) 对文档求加权和(边缘化)。

**怎么做:**
- 定义 `RAGSequence` 类,构造时接收前面创建的 `retriever` 和 `generator`;
- `forward` 方法先用检索器编码问题并取回 top-k 文档及其概率 `doc_probs`;
- 然后遍历每篇检索到的文档:用生成器算出"参考这篇文档时生成整句答案的概率" P(y|x,z),再乘上该文档的检索概率 P(z|x),累加到 `total_prob`——就像请 k 位各拿一份不同资料的专家分别作答,再按对每位专家的信任度做加权平均;
- 最后创建 20 篇随机文档做测试,打印总的 log P(y|x)、实际用到的文档编号 `used_docs` 和各文档权重 `used_probs`。

In [ ]:
class RAGSequence:
    """RAG-Sequence model"""
    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator
    
    def forward(self, query_tokens, target_tokens, document_embeddings, documents_tokens, k=5):
        """
        RAG-Sequence forward pass
        
        P(y|x) = Σ_z P(z|x) * P(y|x,z)
        """
        # Retrieve documents
        # 第一步:检索top-k文档及其概率P(z|x)——文档z是不可观测的隐变量
        query_emb = self.retriever.encode_query(query_tokens)
        doc_indices, doc_probs = self.retriever.retrieve(query_emb, document_embeddings, k=k)

        # Marginalize over documents
        # 边缘化:对隐变量z求和,把"用哪篇文档"的不确定性积掉
        total_prob = 0

        # zip同时遍历文档索引和对应的检索概率
        for doc_idx, p_z_given_x in zip(doc_indices, doc_probs):
            # Get document tokens
            doc_tokens = documents_tokens[doc_idx]
            
            # P(y | x, z)
            # RAG-Sequence特点:整条输出序列的概率都以同一篇文档z为条件
            log_p_y_given_xz = self.generator.generate_prob(query_tokens, doc_tokens, target_tokens)
            p_y_given_xz = np.exp(log_p_y_given_xz)

            # P(z|x) * P(y|x,z)
            # 加权求和实现论文公式P(y|x)=Σ_z P(z|x)P(y|x,z)
            total_prob += p_z_given_x * p_y_given_xz

        return np.log(total_prob + 1e-8), doc_indices, doc_probs

# Create RAG-Sequence model
rag_seq = RAGSequence(retriever, generator)

# Generate dummy documents
num_docs = 20
# 列表推导式生成20篇假文档,每篇是15个token的嵌入矩阵,形状(15, embedding_dim)
documents_tokens = [np.random.randn(15, embedding_dim) for _ in range(num_docs)]

# Test
log_prob, used_docs, used_probs = rag_seq.forward(
    query_tokens=query,
    target_tokens=target,
    document_embeddings=document_embeddings,
    documents_tokens=documents_tokens,
    k=5
)

print("\nRAG-Sequence:")
print(f"Log P(y|x): {log_prob:.4f}")
print(f"Used documents: {used_docs}")
print(f"Document weights: {used_probs}")

## RAG-Token: Marginalize Per Token(RAG-Token:逐 token 边缘化)

$$
P_{RAG-Token}(y | x) = \prod_{i=1}^{|y|} \sum_{z \in \text{top-k}} P(z | x) \cdot P(y_i | x, z, y_{<i})
$$

Can use different documents for different tokens!

不同的 token 可以使用不同的文档!

#### 💻 代码解读

**做什么:** 实现另一种变体 RAG-Token:不是整句答案共用一批文档,而是生成每一个词(token)时都单独对文档做加权,公式为 P(y|x) = ∏_i Σ_z P(z|x)·P(y_i|x,z)。

**怎么做:**
- 定义 `RAGToken` 类,同样组合 `retriever` 和 `generator`;
- `forward_token` 方法只针对一个目标词:检索 top-k 文档,遍历每篇文档算出"参考它生成这个词"的概率,按检索概率加权求和,得到该词的边缘概率——相当于写答案时每写一个词都可以换一份参考资料;
- `forward` 方法遍历答案的所有词,逐个调用 `forward_token`,把每个词概率的对数累加起来(连乘取对数就变成累加),得到整句的 log P(y|x);
- 最后用同样的测试数据运行,打印 RAG-Token 的对数概率,并提示两者的关键区别:RAG-Token 可以在不同词位置使用不同文档。

In [ ]:
class RAGToken:
    """RAG-Token model (simplified)"""
    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator
    
    def forward_token(self, query_tokens, target_token, document_embeddings, documents_tokens, k=5):
        """
        Compute P(y_i | x) for single token
        
        P(y_i | x) = Σ_z P(z|x) * P(y_i|x,z)
        """
        # Retrieve documents
        query_emb = self.retriever.encode_query(query_tokens)
        doc_indices, doc_probs = self.retriever.retrieve(query_emb, document_embeddings, k=k)
        
        # Marginalize for this token
        # 与RAG-Sequence的区别:边缘化发生在单个token级别,而不是整条序列
        token_prob = 0

        for doc_idx, p_z_given_x in zip(doc_indices, doc_probs):
            doc_tokens = documents_tokens[doc_idx]

            # P(y_i | x, z) - simplified
            # 注意target只传入单个token(包成列表),只算这一个token的条件概率
            log_p = self.generator.generate_prob(query_tokens, doc_tokens, [target_token])
            p_yi_given_xz = np.exp(log_p)

            # 每个token都对所有文档加权:P(y_i|x)=Σ_z P(z|x)P(y_i|x,z)
            token_prob += p_z_given_x * p_yi_given_xz
        
        return token_prob, doc_indices, doc_probs
    
    def forward(self, query_tokens, target_tokens, document_embeddings, documents_tokens, k=5):
        """
        Full sequence probability
        
        P(y|x) = ∏_i P(y_i|x)
        """
        log_prob_total = 0

        for target_token in target_tokens:
            # 下划线_表示忽略返回的文档索引和概率,只要token概率
            token_prob, _, _ = self.forward_token(
                query_tokens, target_token, document_embeddings, documents_tokens, k
            )
            # 连乘∏在对数域变成累加,顺便用+1e-8避免log(0)
            log_prob_total += np.log(token_prob + 1e-8)
        
        return log_prob_total

# Create RAG-Token model
rag_token = RAGToken(retriever, generator)

# Test
log_prob_token = rag_token.forward(
    query_tokens=query,
    target_tokens=target,
    document_embeddings=document_embeddings,
    documents_tokens=documents_tokens,
    k=5
)

print("\nRAG-Token:")
print(f"Log P(y|x): {log_prob_token:.4f}")
print("\nDifference: RAG-Token can use different docs per token!")

## Synthetic QA Example(合成问答(QA)示例)

#### 💻 代码解读

**做什么:** 构造一个更贴近现实的迷你问答数据集,直观展示 RAG 在实际场景中的输入长什么样。

**怎么做:**
- 定义 `knowledge_base` 列表:5 条百科式知识文本(埃菲尔铁塔建于 1889 年、巴黎是法国首都、珠穆朗玛峰高 8849 米等),充当"外部知识库";
- 定义 `qa_pairs` 列表:3 个问答三元组,每条包含问题、标准答案以及对应知识条目的编号 `doc_idx`(比如"埃菲尔铁塔何时建成?"的答案 "1889" 来自第 0 条知识);
- 用两个循环把知识库和问答对逐条打印出来,方便对照查看"问题—答案—相关文档"的对应关系。这一格没有计算,纯粹是准备和展示数据。

In [ ]:
# Create more realistic example
knowledge_base = [
    "The Eiffel Tower was built in 1889 by Gustave Eiffel.",
    "Paris is the capital of France and has a population of 2.2 million.",
    "The Statue of Liberty was a gift from France to the United States.",
    "Mount Everest is 8,849 meters tall and located in the Himalayas.",
    "The Amazon River flows through South America for 6,400 kilometers.",
]

# 每个元组是(问题, 答案, 正确文档的索引),索引用于检查检索是否命中
qa_pairs = [
    ("When was the Eiffel Tower built?", "1889", 0),
    ("What is the height of Mount Everest?", "8,849 meters", 3),
    ("How long is the Amazon River?", "6,400 kilometers", 4),
]

print("Knowledge Base:")
for i, doc in enumerate(knowledge_base):
    print(f"  {i}. {doc}")

print("\nQA Pairs:")
# 元组解包:循环时直接把三元组拆成问题、答案、文档索引
for q, a, doc_idx in qa_pairs:
    print(f"  Q: {q}")
    print(f"  A: {a}")
    print(f"  Relevant doc: #{doc_idx}")
    print()

## Visualize RAG Architecture(可视化 RAG 架构)

#### 💻 代码解读

**做什么:** 用 matplotlib 手工画出两张架构示意图,左右并排对比 RAG-Sequence 和 RAG-Token 的信息流动方式。

**怎么做:**
- 用 `plt.subplots(1, 2)` 创建左右两个画布;
- 定义 `draw_rag_variant` 函数,像搭积木一样用 `Rectangle`(方框)、`text`(文字)和 `arrow`(箭头)画出流程:顶部是查询 Query (x),往下是检索器 Retriever (DPR),再往下是取回的 Top-k 文档 z1、z2、z3;
- 当参数 `is_token=False`(RAG-Sequence)时:画出每篇文档各自接一个生成器 Gen、各自生成完整答案 y,最后汇入 "Σ P(z|x)P(y|x,z)" 的加权求和框——一篇资料写一整份答案再综合;
- 当 `is_token=True`(RAG-Token)时:画出每个输出词 y1、y2、y3 都从所有文档收到虚线箭头,汇入 "∏ Σ P(z|x)P(yi|x,z)" 框——每个词都参考所有资料;
- 两条路径最终都指向底部绿色的 Answer 框,`plt.show()` 显示图像。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def draw_rag_variant(ax, title, is_token=False):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Query
    ax.add_patch(plt.Rectangle((4, 10.5), 2, 0.8, fill=True, 
                               color='lightblue', ec='black', linewidth=2))
    ax.text(5, 10.9, 'Query (x)', ha='center', va='center', fontsize=11, fontweight='bold')
    
    # Retriever
    ax.add_patch(plt.Rectangle((3.5, 9), 3, 1, fill=True, 
                               color='lightgreen', ec='black', linewidth=2))
    ax.text(5, 9.5, 'Retriever\n(DPR)', ha='center', va='center', fontsize=10, fontweight='bold')
    ax.arrow(5, 10.5, 0, -0.3, head_width=0.2, head_length=0.1, fc='black', ec='black', linewidth=2)
    
    # Retrieved documents
    doc_positions = [2, 4, 6, 8]
    ax.text(5, 7.8, 'Top-k Documents', ha='center', fontsize=10, fontweight='bold')
    for i, x in enumerate(doc_positions[:3]):
        ax.add_patch(plt.Rectangle((x-0.4, 6.5), 0.8, 1, fill=True, 
                                   color='lightyellow', ec='black', linewidth=1.5))
        ax.text(x, 7, f'z{i+1}', ha='center', va='center', fontsize=9)
        # Arrow from retriever
        ax.plot([5, x], [9, 7.5], 'k--', alpha=0.5, linewidth=1)
    
    # 用同一个绘图函数画两种变体,is_token开关决定画哪种边缘化方式
    if not is_token:
        # RAG-Sequence: each doc generates full sequence
        y_positions = [2, 4, 6]
        # zip配对文档位置与输出位置,enumerate同时给出序号i
        for i, (dx, dy) in enumerate(zip(doc_positions[:3], y_positions)):
            # Generator per document
            ax.add_patch(plt.Rectangle((dy-0.5, 4.5), 1, 0.8, fill=True, 
                                       color='lightcoral', ec='black', linewidth=1.5))
            ax.text(dy, 4.9, f'Gen', ha='center', va='center', fontsize=8)
            ax.arrow(dx, 6.5, dy-dx, -1.5, head_width=0.15, head_length=0.1, 
                    fc='gray', ec='gray', linewidth=1, alpha=0.6)
            
            # Output sequence
            ax.add_patch(plt.Rectangle((dy-0.6, 3), 1.2, 0.6, fill=True, 
                                       color='wheat', ec='black', linewidth=1))
            ax.text(dy, 3.3, f'y', ha='center', va='center', fontsize=8)
            ax.arrow(dy, 4.5, 0, -0.8, head_width=0.12, head_length=0.08, 
                    fc='black', ec='black', linewidth=1)
        
        # Combine
        ax.add_patch(plt.Rectangle((4, 1.2), 2, 0.8, fill=True, 
                                   color='plum', ec='black', linewidth=2))
        ax.text(5, 1.6, 'Σ P(z|x)P(y|x,z)', ha='center', va='center', fontsize=9, fontweight='bold')
        for dy in y_positions:
            ax.plot([dy, 5], [3, 2], 'k-', alpha=0.5, linewidth=1.5)
    else:
        # RAG-Token: combine docs for each token
        # 每个输出token都连到所有文档,体现"逐token混合不同知识来源"
        token_y = 4.5
        for t in range(3):
            tx = 2 + t * 2.5
            
            # Token position
            ax.add_patch(plt.Rectangle((tx-0.4, token_y), 0.8, 0.6, fill=True, 
                                       color='lightcoral', ec='black', linewidth=1.5))
            ax.text(tx, token_y+0.3, f'y{t+1}', ha='center', va='center', fontsize=9)
            
            # Arrows from all docs
            for dx in doc_positions[:3]:
                ax.plot([dx, tx], [6.5, token_y+0.6], 'k--', alpha=0.3, linewidth=0.8)
        
        # Final output
        ax.add_patch(plt.Rectangle((3.5, 2.5), 3, 0.8, fill=True, 
                                   color='plum', ec='black', linewidth=2))
        ax.text(5, 2.9, '∏ Σ P(z|x)P(yi|x,z)', ha='center', va='center', 
               fontsize=9, fontweight='bold')
        ax.arrow(4, token_y, 0.8, -1.3, head_width=0.15, head_length=0.1, 
                fc='black', ec='black', linewidth=1.5, alpha=0.5)
    
    # Final answer
    ax.add_patch(plt.Rectangle((4, 0.3), 2, 0.6, fill=True, 
                               color='lightgreen', ec='black', linewidth=2))
    ax.text(5, 0.6, 'Answer', ha='center', va='center', fontsize=11, fontweight='bold')
    ax.arrow(5, 1.2 if not is_token else 2.5, 0, 
            -0.2 if not is_token else -1.5, 
            head_width=0.2, head_length=0.1, fc='green', ec='green', linewidth=2)

draw_rag_variant(axes[0], 'RAG-Sequence', is_token=False)
draw_rag_variant(axes[1], 'RAG-Token', is_token=True)

plt.tight_layout()
plt.show()

## Compare RAG Variants(比较 RAG 变体)

#### 💻 代码解读

**做什么:** 用两张热力图直观对比两种 RAG 变体的文档权重分布:RAG-Sequence 所有词共用一套权重,RAG-Token 每个词各有一套权重。

**怎么做:**
- 设定 5 篇文档(`n_docs`)、8 个输出词(`n_tokens`);
- RAG-Sequence 一侧:用 `softmax` 生成一组文档权重 `doc_weights_seq`,再用 `np.tile` 把这一行复制 8 遍得到 `weights_seq_matrix`——好比整篇作文从头到尾只信同一批参考书;
- RAG-Token 一侧:循环 8 次,每次独立生成一组 softmax 权重,组成 `weights_token_matrix`——每写一个词就重新掂量一遍各文档的重要性;
- 用 `imshow` 把两个矩阵画成热力图(横轴是词的位置,纵轴是文档,颜色越深权重越大),并加上颜色条标注 P(z|x);
- 最后打印总结:RAG-Sequence 更一致(始终用同一批知识),RAG-Token 更灵活(可以混合多个知识来源)。

In [ ]:
# Simulate probabilities for visualization
n_docs = 5
n_tokens = 8

# RAG-Sequence: same doc weights for all tokens
doc_weights_seq = softmax(np.random.randn(n_docs))
# np.tile把同一组文档权重沿行方向复制n_tokens份:(n_docs,) -> (n_tokens, n_docs)
weights_seq_matrix = np.tile(doc_weights_seq, (n_tokens, 1))

# RAG-Token: different doc weights per token
# 列表推导式为每个token独立采样一组softmax权重,得到(n_tokens, n_docs)矩阵
weights_token_matrix = np.array([softmax(np.random.randn(n_docs)) for _ in range(n_tokens)])

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# .T转置后画热力图:行是文档、列是token位置;固定vmin/vmax让两幅图颜色可比
im1 = ax1.imshow(weights_seq_matrix.T, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax1.set_xlabel('Output Token Position', fontsize=12)
ax1.set_ylabel('Document', fontsize=12)
ax1.set_title('RAG-Sequence\n(Same docs for all tokens)', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=ax1, label='P(z|x)')

im2 = ax2.imshow(weights_token_matrix.T, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax2.set_xlabel('Output Token Position', fontsize=12)
ax2.set_ylabel('Document', fontsize=12)
ax2.set_title('RAG-Token\n(Different docs per token)', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=ax2, label='P(z|x)')

plt.tight_layout()
plt.show()

print("\nRAG-Sequence: More consistent (uses same knowledge)")
print("RAG-Token: More flexible (can mix knowledge sources)")

## Key Takeaways(关键要点)

### RAG Architecture:(RAG 架构:)

**Components**:
1. **Retriever**: Dense retrieval (DPR-style)
   - Query encoder: $q_{emb} = E_Q(x)$
   - Document encoder: $d_{emb} = E_D(z)$
   - Retrieval: $P(z|x) \propto \exp(q_{emb} \cdot d_{emb})$

2. **Generator**: Seq2seq model (BART)
   - Input: query $x$ + document $z$
   - Output: $P(y | x, z)$

**组件**:
1. **检索器(Retriever)**:稠密检索(DPR 风格)
   - 查询编码器:$q_{emb} = E_Q(x)$
   - 文档编码器:$d_{emb} = E_D(z)$
   - 检索:$P(z|x) \propto \exp(q_{emb} \cdot d_{emb})$

2. **生成器(Generator)**:序列到序列模型(BART)
   - 输入:查询 $x$ + 文档 $z$
   - 输出:$P(y | x, z)$

### RAG-Sequence:

$$
P_{RAG-Seq}(y | x) = \sum_{z \in \text{top-k}} P(z | x) \cdot P_{seq2seq}(y | x, z)
$$

**Process**:
1. Retrieve top-k documents
2. Generate full sequence with each document
3. Weighted sum of sequences

**Characteristics**:
- Each document generates complete answer
- More consistent (single knowledge source per sequence)
- Better for factoid QA

**流程**:
1. 检索 top-k 篇文档
2. 用每篇文档分别生成完整序列
3. 对各序列做加权求和

**特点**:
- 每篇文档都生成完整答案
- 更一致(每个序列只用单一知识来源)
- 更适合事实型问答(factoid QA)

### RAG-Token:

$$
P_{RAG-Token}(y | x) = \prod_{i=1}^{|y|} \left( \sum_{z \in \text{top-k}} P(z | x) \cdot P(y_i | x, z, y_{<i}) \right)
$$

**Process**:
1. Retrieve top-k documents (same for all tokens)
2. For each token: marginalize over documents
3. Different documents can contribute to different tokens

**Characteristics**:
- Can mix information from multiple documents
- More flexible generation
- Better for long-form generation

**流程**:
1. 检索 top-k 篇文档(所有 token 共用)
2. 对每个 token:在文档上做边缘化
3. 不同文档可以对不同 token 做出贡献

**特点**:
- 可以混合来自多篇文档的信息
- 生成更灵活
- 更适合长文本生成

### Training:(训练:)

**End-to-end**:
```
Loss = -log P(y* | x)
```

Gradients flow through:
- Generator (BART parameters)
- Query encoder (retriever parameters)

**Document encoder**: Usually frozen (pre-indexed)

**端到端(End-to-end)**:梯度流经:
- 生成器(BART 参数)
- 查询编码器(检索器参数)

**文档编码器**:通常冻结(预先建好索引)

### Implementation Details:(实现细节:)

**From paper**:
- Retriever: DPR with BERT-base
- Generator: BART-large (400M params)
- Knowledge: Wikipedia (21M passages)
- Top-k: k=5 or k=10
- Index: FAISS for fast retrieval

**论文中的设置**:
- 检索器:基于 BERT-base 的 DPR
- 生成器:BART-large(4 亿参数)
- 知识库:维基百科(2100 万个段落)
- Top-k:k=5 或 k=10
- 索引:使用 FAISS 进行快速检索

### Results:(结果:)

**Natural Questions (Open)**:
- BART (no retrieval): 27.0% EM
- RAG-Sequence: 44.5% EM
- RAG-Token: 44.1% EM

**TriviaQA**:
- BART: 50.1%
- RAG: 56.8%

**WebQuestions**:
- BART: 27.6%
- RAG: 45.2%

**Natural Questions(开放域)**:
- BART(无检索):27.0% EM
- RAG-Sequence:44.5% EM
- RAG-Token:44.1% EM

**TriviaQA**:
- BART:50.1%
- RAG:56.8%

**WebQuestions**:
- BART:27.6%
- RAG:45.2%

### RAG vs Baselines:(RAG 与基线对比:)

| Model | Knowledge | Parametric | Performance |
|-------|-----------|------------|-------------|
| T5-11B | Memorized | ✓ | Good |
| REALM | Retrieved | Mixed | Better |
| **RAG** | **Retrieved** | **✓** | **Best** |

| 模型 | 知识来源 | 参数化 | 性能 |
|-------|-----------|------------|-------------|
| T5-11B | 记忆(参数内) | ✓ | 好 |
| REALM | 检索 | 混合 | 更好 |
| **RAG** | **检索** | **✓** | **最佳** |

### Advantages:(优点:)

- ✅ **Factual accuracy**: Access to external knowledge
- ✅ **Scalability**: Add knowledge without retraining
- ✅ **Interpretability**: Can inspect retrieved documents
- ✅ **Efficiency**: Smaller models than pure parametric
- ✅ **Up-to-date**: Update index, not model weights

- ✅ **事实准确性**:可访问外部知识
- ✅ **可扩展性**:无需重新训练即可添加知识
- ✅ **可解释性**:可以检查检索到的文档
- ✅ **高效**:比纯参数化模型更小
- ✅ **时效性**:更新索引即可,无需更新模型权重

### Limitations:(局限性:)

- ❌ **Retrieval errors**: Wrong docs → wrong answers
- ❌ **Latency**: Retrieval adds overhead
- ❌ **Index maintenance**: Need to re-encode for updates
- ❌ **Memory**: Full document index required

- ❌ **检索错误**:检索到错误文档 → 得到错误答案
- ❌ **延迟**:检索带来额外开销
- ❌ **索引维护**:更新时需要重新编码
- ❌ **内存**:需要保存完整的文档索引

### When to Use:(何时使用:)

**RAG-Sequence**:
- Factoid QA
- Short answers
- When single source is enough

**RAG-Token**:
- Long-form generation
- Multi-hop reasoning
- Combining multiple sources

**RAG-Sequence**:
- 事实型问答
- 简短答案
- 单一来源即可满足需求时

**RAG-Token**:
- 长文本生成
- 多跳推理(multi-hop reasoning)
- 需要组合多个来源时

### Modern Extensions:(现代扩展:)

- **RETRO** (DeepMind): Retrieve at every layer
- **Atlas** (Meta): Improved training
- **Toolformer**: Retrieve via API calls
- **WebGPT**: Interactive retrieval
- **Self-RAG**: Self-reflective retrieval

- **RETRO**(DeepMind):在每一层进行检索
- **Atlas**(Meta):改进的训练方式
- **Toolformer**:通过 API 调用进行检索
- **WebGPT**:交互式检索
- **Self-RAG**:自我反思式检索

### Production Tips:(生产实践建议:)

1. **Hybrid ranking**: Combine retrieval + reranking
2. **Cache**: Pre-retrieve for common queries
3. **Async**: Retrieve while generating
4. **Fallback**: Parametric generation if retrieval fails
5. **Monitor**: Track retrieval quality

1. **混合排序**:结合检索 + 重排序(reranking)
2. **缓存**:为常见查询预先检索
3. **异步**:边生成边检索
4. **兜底**:检索失败时回退到参数化生成
5. **监控**:持续跟踪检索质量

### Applications:(应用:)

- Open-domain QA (Google, Bing)
- Chatbots with knowledge bases
- Document QA
- Fact-checking
- Research assistants
- Customer support

- 开放域问答(Google、Bing)
- 带知识库的聊天机器人
- 文档问答
- 事实核查
- 研究助手
- 客户支持

### Key Insight:(核心洞见:)

**RAG = Best of both worlds**
- Parametric knowledge (generation capability)
- Non-parametric knowledge (external retrieval)
- End-to-end differentiable
- Practical and effective!

**RAG = 两全其美**
- 参数化知识(生成能力)
- 非参数化知识(外部检索)
- 端到端可微分
- 实用且有效!